# CS4168 Data Mining — Spotify Tracks Analysis

## Introduction

**Students:** Alexey Sidorov (22349073), Liam Kelly (22346317), Evan Buggy (22348069)

This notebook presents a data mining pipeline for the **tracks2026.csv** dataset, containing Spotify track metadata and audio features. We will:

1. **Explore** the data (EDA) and visualize key patterns  
2. **Preprocess** the data for modeling  
3. **Cluster** tracks using K-Means and inspect genre correspondence  
4. **Classify** tracks as above/below median popularity  
5. **Predict** popularity with regression models  

We conclude with a brief discussion of findings and limitations.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
sns.set_style("whitegrid")

In [ ]:
# Load dataset (same directory as notebook)
df = pd.read_csv("tracks2026.csv")
df.head(10)

## Data Exploration

We inspect the dataset shape, columns, missing values, and summary statistics.

In [ ]:
# Shape and columns
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:\n", df.dtypes)

In [ ]:
# Missing values
missing = df.isnull().sum()
print("Missing values per column:\n", missing[missing > 0].sort_values(ascending=False))
if missing.sum() == 0:
    print("No missing values.")
else:
    print("\nTotal missing:", missing.sum())

In [ ]:
# Summary statistics
df.describe()

### Visualizations

We create at least five visualizations: histograms of numeric features, correlation heatmap, popularity distribution, feature relationships, and genre distribution.

In [ ]:
# 1. Histograms of key numeric features
numeric_cols = ['popularity', 'danceability', 'energy', 'loudness', 'acousticness', 'valence', 'tempo']
df_numeric = df[numeric_cols].dropna(how='all', axis=1)
df_numeric.hist(bins=30, figsize=(12, 8), edgecolor='black', alpha=0.8)
plt.suptitle("Histograms of Numeric Features", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Correlation heatmap (numeric columns only)
numeric_for_corr = df.select_dtypes(include=[np.number]).dropna(axis=0, how='any')
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_for_corr.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Popularity distribution
plt.figure(figsize=(8, 5))
df['popularity'].dropna().hist(bins=40, edgecolor='black', alpha=0.8)
plt.xlabel("Popularity")
plt.ylabel("Count")
plt.title("Distribution of Track Popularity")
plt.show()

In [ ]:
# 4. Feature relationships: scatter matrix (sample for speed) or key pairs
# Energy vs danceability colored by popularity (binned)
df_plot = df.dropna(subset=['energy', 'danceability', 'popularity'])
df_plot = df_plot.sample(min(800, len(df_plot)), random_state=42)
plt.figure(figsize=(8, 5))
sc = plt.scatter(df_plot['danceability'], df_plot['energy'], c=df_plot['popularity'], cmap='viridis', alpha=0.6)
plt.colorbar(sc, label='Popularity')
plt.xlabel("Danceability")
plt.ylabel("Energy")
plt.title("Danceability vs Energy (colored by Popularity)")
plt.show()

In [ ]:
# 5. Genre distribution
genre_counts = df['track_genre'].value_counts()
plt.figure(figsize=(10, 5))
genre_counts.plot(kind='bar', edgecolor='black', alpha=0.8)
plt.xlabel("Genre")
plt.ylabel("Count")
plt.title("Distribution of Track Genres")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**How EDA informs later tasks:** The EDA above informs (1) **preprocessing:** we drop rows with missing values in key columns and scale numeric features for clustering and for linear models; (2) **clustering:** we use only numeric audio features (no genre) so clusters reflect acoustic similarity; (3) **classification/regression:** correlations and distributions suggest which features may predict popularity (e.g. energy, loudness), and the balanced genre distribution supports using the same preprocessing across the dataset.

## Data Preprocessing

We drop non-numeric columns where needed for modeling, handle missing values, and normalize/scale numerical features. We keep a copy of the original dataframe for reference and genre labels.

In [ ]:
# Keep genre and track_id for interpretation; work on a copy
df_clean = df.copy()

# Convert explicit from "TRUE"/"FALSE" to 0/1 if needed
if 'explicit' in df_clean.columns and df_clean['explicit'].dtype == object:
    df_clean['explicit'] = (df_clean['explicit'].astype(str).str.upper() == 'TRUE').astype(int)

# Select numeric columns for modeling (excluding identifiers)
feature_cols = [c for c in df_clean.columns if c not in ['track_id', 'track_genre'] and np.issubdtype(df_clean[c].dtype, np.number)]
print("Numeric feature columns:", feature_cols)

In [ ]:
# Drop rows with missing values in key numeric columns so models can run
df_clean = df_clean.dropna(subset=feature_cols + ['popularity'])
print("Shape after dropping rows with missing values:", df_clean.shape)
print("Missing values remaining:", df_clean[feature_cols].isnull().sum().sum())

In [ ]:
# Normalize/scale numerical features (for clustering and optional use in classification/regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[feature_cols])
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=df_clean.index)
print("Scaled feature matrix shape:", X_scaled_df.shape)
X_scaled_df.head()

## Clustering

We apply **K-Means** on the scaled numeric features (excluding the target). We use the **elbow method** to choose \(k\), then visualize clusters in 2D using **PCA**.

In [ ]:
# For clustering, use features only (exclude popularity so clusters reflect audio characteristics)
clustering_cols = [c for c in feature_cols if c != 'popularity']
X_cluster = scaler.fit_transform(df_clean[clustering_cols])

### Ensure track_genre is excluded

**track_genre** must not be used for clustering so that clusters are based only on audio features, not on the genre label.

In [ ]:
# Ensure track_genre is excluded from clustering features (use only numeric audio features)
clustering_cols = [c for c in clustering_cols if c != 'track_genre']
X_cluster = scaler.fit_transform(df_clean[clustering_cols])

In [ ]:
# Elbow method to choose k
inertias = []
K_range = range(2, 16)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal k")
plt.xticks(K_range)
plt.show()

Compare clustering quality for **multiple K values** (3, 5, and 8) by reporting inertia; lower inertia means tighter clusters.

In [ ]:
for k in [3, 5, 8]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    print(f"k={k}, inertia={km.inertia_}")

**Final clustering solution:** We select **K-Means with k=5** as the final clustering solution. The elbow plot shows a bend in inertia around k=4–6, and among k=3, 5, and 8, k=5 offers a good trade-off: fewer clusters than k=8 (easier to interpret) while capturing more structure than k=3. We use this solution for the cluster–genre analysis and visualisation below.

In [ ]:
# Fit K-Means with chosen k (e.g. k=5 from elbow; adjust if elbow suggests different)
k_optimal = 5
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster)
df_clean = df_clean.copy()
df_clean['cluster'] = cluster_labels

In [ ]:
# Dimensionality reduction with PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)
print("PCA explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained:", pca.explained_variance_ratio_.sum().round(4))

In [ ]:
# Visualize clusters in PCA space
plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='tab10', alpha=0.6)
plt.colorbar(scatter, label='Cluster')
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-Means Clusters (PCA projection)")
plt.show()

**DBSCAN** is a density-based clustering method that can detect noise (points labeled -1) and does not require specifying the number of clusters. We use the same scaled data and PCA projection for visualization.

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.8, min_samples=5)
db_labels = db.fit_predict(X_cluster)

plt.figure(figsize=(9, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=db_labels, cmap='tab10', alpha=0.6)
plt.colorbar(label="Cluster")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("DBSCAN Clusters (PCA Projection)")
plt.show()

In [ ]:
# Do clusters correspond to genres? Cross-tabulation
ct = pd.crosstab(df_clean['cluster'], df_clean['track_genre'])
print("Cluster vs Genre counts:\n", ct)
# Normalized by row to see genre mix per cluster
ct_norm = ct.div(ct.sum(axis=1), axis=0)
print("\nGenre proportion within each cluster (row-normalized):\n", ct_norm.round(3))

**Brief interpretation:** Clusters are based on audio features (danceability, energy, etc.), not on genre labels. So we do *not* expect a one-to-one match between clusters and genres. If some clusters have a higher proportion of a certain genre, it suggests that genre tends to have a distinct audio profile; otherwise, genres are mixed within clusters, meaning different genres can share similar audio features.

**Why clustering succeeded or failed:** Clustering **succeeded** in revealing meaningful structure in terms of audio similarity (tracks in the same cluster share similar danceability, energy, etc.). It **failed** to align with genre labels because we deliberately excluded `track_genre` from the features; genres are mixed within clusters, which is expected and shows that different genres can share similar acoustic profiles.

**Pipeline (preprocessing + model):** The spec requires preprocessing inside a proper machine learning pipeline. A full pipeline example (Scaler + Logistic Regression) is included in the **Classification** section below, immediately after the train/test split so that `X_train` and `y_train` are defined.

In [ ]:
# Pipeline example (Scaler + Logistic Regression) is in the Classification section below,
# right after the train_test_split cell, so that X_train, y_train are defined. Run cells in order.

## Classification

We create a binary target **popularity_binary** (1 if popularity > median, 0 otherwise), then train a classifier (Random Forest or Logistic Regression), split into train/test, and report accuracy, confusion matrix, and classification report.

In [ ]:
# Create popularity_binary: 1 if popularity > median, else 0
df_clean = df_clean.copy()  # ensure we have a copy before adding column
median_pop = df_clean['popularity'].median()
df_clean['popularity_binary'] = (df_clean['popularity'] > median_pop).astype(int)
print("Median popularity:", median_pop)
print("Class counts:\n", df_clean['popularity_binary'].value_counts())

In [ ]:
# Features for classification: use numeric features excluding the target (popularity)
X_class_cols = [c for c in feature_cols if c != 'popularity']
X_clf = df_clean[X_class_cols]
y_clf = df_clean['popularity_binary']

X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.25, random_state=42, stratify=y_clf)
# Scale features for Logistic Regression; Random Forest is scale-invariant
scaler_clf = StandardScaler()
X_train_s = scaler_clf.fit_transform(X_train)
X_test_s = scaler_clf.transform(X_test)

**Pipeline (preprocessing + model):** Preprocessing is done inside an `sklearn.pipeline` so that scaling is fitted on the training data only and applied consistently at prediction time (e.g. for cross-validation).

In [ ]:
# Example: classification pipeline (StandardScaler + Logistic Regression)
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
pipe_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=42))
pipe_lr.fit(X_train, y_train)
print("Pipeline (Scaler + LR) test accuracy:", round(accuracy_score(y_test, pipe_lr.predict(X_test)), 4))
# Cross-validation with pipeline ensures scaling is refit per fold
scores_pipe = cross_val_score(pipe_lr, X_clf, y_clf, cv=5)
print("Pipeline 5-fold CV accuracy: mean =", round(scores_pipe.mean(), 4))

In [ ]:
# Train Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)  # no scaling needed for RF
y_pred_rf = clf.predict(X_test)
print("Random Forest - Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

**Cross-validation** gives a more stable estimate of classifier performance by training and evaluating on multiple train/test splits (here, 5-fold). It helps ensure the model generalizes well beyond a single train/test split.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(clf, X_clf, y_clf, cv=5)
print("Cross-validation accuracy scores:", scores)
print("Mean CV accuracy:", scores.mean())

**Feature importance** from the Random Forest shows which audio features contribute most to predicting above/below median popularity. This helps interpret which audio features contribute most to popularity prediction.

In [ ]:
importances = pd.Series(clf.feature_importances_, index=X_class_cols)

plt.figure(figsize=(8, 5))
importances.sort_values().plot(kind="barh")
plt.title("Feature Importance for Predicting Popularity")
plt.xlabel("Importance")
plt.show()

In [ ]:
# Also train Logistic Regression (on scaled features)
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
print("Logistic Regression - Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

**Final classification model:** We select **Random Forest** as the final classifier. We use **accuracy** (and the classification report) for model selection because we have a balanced binary target and care about overall correctness; cross-validation accuracy is more stable than a single split. Random Forest achieved higher accuracy than Logistic Regression on the test set and in cross-validation, and it does not require feature scaling while still capturing non-linear relationships.

## Regression

We predict the continuous **popularity** score using at least one regression model (Linear Regression and/or Random Forest Regressor), and evaluate with **RMSE** and **R²**.

In [ ]:
# Regression: predict popularity (use same feature set as classification, defined here for independence)
regression_cols = [c for c in feature_cols if c != 'popularity']
X_reg = df_clean[regression_cols]
y_reg = df_clean['popularity']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.25, random_state=42)
scaler_reg = StandardScaler()
X_train_rs = scaler_reg.fit_transform(X_train_r)
X_test_rs = scaler_reg.transform(X_test_r)

In [ ]:
# Linear Regression (use arrays to avoid index alignment issues)
lm = LinearRegression()
lm.fit(X_train_rs, np.asarray(y_train_r))
y_pred_lm = lm.predict(X_test_rs)
rmse_lm = np.sqrt(mean_squared_error(y_test_r, y_pred_lm))
r2_lm = r2_score(y_test_r, y_pred_lm)
print("Linear Regression:")
print("  RMSE:", round(rmse_lm, 4))
print("  R²:", round(r2_lm, 4))

In [ ]:
# Random Forest Regressor
rfr = RandomForestRegressor(n_estimators=100, random_state=42)
rfr.fit(X_train_r, y_train_r)  # no scaling for RF
y_pred_rfr = rfr.predict(X_test_r)
rmse_rfr = np.sqrt(mean_squared_error(y_test_r, y_pred_rfr))
r2_rfr = r2_score(y_test_r, y_pred_rfr)
print("Random Forest Regressor:")
print("  RMSE:", round(rmse_rfr, 4))
print("  R²:", round(r2_rfr, 4))

## Discussion and Conclusions

- **EDA:** The dataset contains track-level audio features and popularity; we observed distributions of key variables, correlations, and genre balance.
- **Preprocessing:** Non-numeric identifiers were excluded from the feature set; missing values were handled by dropping incomplete rows; numeric features were scaled for clustering and for Linear/Logistic models.
- **Clustering:** K-Means on audio features (with \(k\) chosen by the elbow method) groups tracks by acoustic similarity. Clusters do not necessarily align with genre labels, since genre is a label and clusters are driven only by the numeric features.
- **Classification:** Predicting above/below median popularity is feasible with Random Forest and Logistic Regression; accuracy and class-wise metrics depend on the train/test split and feature set.
- **Regression:** Predicting exact popularity is harder (modest R² is common); Random Forest often outperforms Linear Regression when relationships are non-linear.
- **Limitations:** Dropping missing rows may introduce bias; popularity is influenced by factors outside this dataset (e.g. playlist placement, marketing); results are specific to this sample and time period.

**Final regression model and metrics:** We select **Random Forest Regressor** as the final regression model because it achieved lower RMSE and higher R² than Linear Regression. We use **RMSE** (scale of the target) and **R²** (variance explained) for model selection; both are standard for regression. Below we add cross-validation for regression and feature importance to support this choice.

In [ ]:
# Cross-validation for regression (5-fold) using pipelines so scaling is per fold
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
pipe_lm = make_pipeline(StandardScaler(), LinearRegression())
pipe_rfr = make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42))
cv_rmse_lm = np.sqrt(-cross_val_score(pipe_lm, X_train_r, y_train_r, cv=5, scoring='neg_mean_squared_error'))
cv_r2_lm = cross_val_score(pipe_lm, X_train_r, y_train_r, cv=5, scoring='r2')
cv_rmse_rfr = np.sqrt(-cross_val_score(pipe_rfr, X_train_r, y_train_r, cv=5, scoring='neg_mean_squared_error'))
cv_r2_rfr = cross_val_score(pipe_rfr, X_train_r, y_train_r, cv=5, scoring='r2')
print("Linear Regression  CV RMSE: mean =", round(cv_rmse_lm.mean(), 4), ", std =", round(cv_rmse_lm.std(), 4))
print("Linear Regression  CV R²:  mean =", round(cv_r2_lm.mean(), 4))
print("Random Forest Reg  CV RMSE: mean =", round(cv_rmse_rfr.mean(), 4), ", std =", round(cv_rmse_rfr.std(), 4))
print("Random Forest Reg  CV R²:  mean =", round(cv_r2_rfr.mean(), 4))

In [ ]:
# Feature importance for regression (Random Forest Regressor)
imp_reg = pd.Series(rfr.feature_importances_, index=regression_cols)
plt.figure(figsize=(8, 5))
imp_reg.sort_values().plot(kind="barh")
plt.title("Feature Importance for Predicting Popularity (Regression)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

**Summary.** Clustering (K-Means and DBSCAN) on audio features showed that tracks group by acoustic similarity rather than by genre, so different genres can share similar sound profiles. Classifying above/below median popularity achieved moderate accuracy (e.g. with Random Forest and cross-validation), indicating that audio features carry some signal but popularity is only partly predictable from them. Regression to predict exact popularity gave modest R², with Random Forest typically doing better than Linear Regression, which reflects that popularity depends on many factors beyond the dataset (e.g. playlists and marketing). Limitations include missing values handled by dropping rows, the fact that popularity is influenced by external factors not in the data, and that results apply to this sample and time period only.